# Build and Tune Your Own Sequence Aligner

Module 2 Project - CMU Pre-College Computational Biology

This notebook contains a small alignment app that can switch between global and local alignment, tune match/mismatch/gap scoring, compare DNA and protein alignments, and record a good-versus-bad parameter story.

The dynamic programming alignment code below is written directly in this notebook. Biopython is used only to load standard protein scoring matrices when available.

## Quick start

1. Click **Runtime > Run all** in Colab.
2. Stop at the **Interactive Alignment App** section.
3. Choose a preset or paste your own sequences.
4. Change the scoring sliders and click **Run alignment**.
5. Use the colored output and readouts to explain why one parameter set is biologically better than another.

## 1. Setup

In [ ]:
import re
import textwrap
import urllib.parse
import urllib.request
from dataclasses import dataclass

import numpy as np
from IPython.display import HTML, display

try:
    import ipywidgets as widgets
    from ipywidgets import interact
except Exception as exc:
    widgets = None
    print('ipywidgets is not available:', exc)

try:
    from Bio.Align import substitution_matrices
except Exception:
    try:
        import subprocess, sys
        print('Installing Biopython so BLOSUM62/PAM250 protein matrices are available...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'biopython'])
        from Bio.Align import substitution_matrices
    except Exception:
        substitution_matrices = None
        print('Biopython is not installed. DNA/simple scoring will work; protein matrices need Biopython.')

## 2. Alignment Engine

Directions in the traceback matrix:

- `1` = diagonal, aligning one character from each sequence
- `2` = up, putting a gap in sequence 2
- `3` = left, putting a gap in sequence 1
- `0` = stop, used only by local alignment

In [ ]:
@dataclass
class AlignmentResult:
    aligned_a: str
    aligned_b: str
    score: int
    mode: str
    start_a: int
    end_a: int
    start_b: int
    end_b: int


def clean_sequence(seq):
    return re.sub(r'[^A-Za-z*]', '', seq).upper()


def make_score_function(seq_type='DNA', match=1, mismatch=-1, matrix_name='Simple'):
    seq_type = seq_type.upper()
    if seq_type == 'PROTEIN' and matrix_name != 'Simple':
        if substitution_matrices is None:
            raise RuntimeError('Install Biopython or choose the Simple matrix.')
        matrix = substitution_matrices.load(matrix_name)
        def score(x, y):
            try:
                return int(matrix[x, y])
            except Exception:
                return mismatch
        return score

    def simple_score(x, y):
        return match if x == y else mismatch
    return simple_score


def align(seq_a, seq_b, mode='global', seq_type='DNA', match=1, mismatch=-1, gap=-2, matrix_name='Simple'):
    a = clean_sequence(seq_a)
    b = clean_sequence(seq_b)
    if not a or not b:
        raise ValueError('Both sequences must contain at least one letter.')

    mode = mode.lower()
    if mode not in {'global', 'local'}:
        raise ValueError("mode must be 'global' or 'local'")

    score_chars = make_score_function(seq_type, match, mismatch, matrix_name)
    n, m = len(a), len(b)
    scores = np.zeros((n + 1, m + 1), dtype=np.int32)
    trace = np.zeros((n + 1, m + 1), dtype=np.int8)

    if mode == 'global':
        scores[:, 0] = np.arange(n + 1, dtype=np.int32) * gap
        scores[0, :] = np.arange(m + 1, dtype=np.int32) * gap
        trace[1:, 0] = 2
        trace[0, 1:] = 3

    best_i, best_j, best_score = n, m, int(scores[n, m])

    for i in range(1, n + 1):
        ai = a[i - 1]
        for j in range(1, m + 1):
            diagonal = scores[i - 1, j - 1] + score_chars(ai, b[j - 1])
            up = scores[i - 1, j] + gap
            left = scores[i, j - 1] + gap

            if mode == 'local':
                candidates = (0, diagonal, up, left)
                best = max(candidates)
                direction = candidates.index(best)
            else:
                candidates = (diagonal, up, left)
                best = max(candidates)
                direction = candidates.index(best) + 1

            scores[i, j] = best
            trace[i, j] = direction

            if mode == 'local' and best > best_score:
                best_i, best_j, best_score = i, j, int(best)

    if mode == 'global':
        best_i, best_j, best_score = n, m, int(scores[n, m])

    aligned_a = []
    aligned_b = []
    i, j = best_i, best_j
    end_a, end_b = i, j

    while i > 0 or j > 0:
        direction = int(trace[i, j])
        if mode == 'local' and direction == 0:
            break
        if direction == 1:
            aligned_a.append(a[i - 1])
            aligned_b.append(b[j - 1])
            i -= 1
            j -= 1
        elif direction == 2:
            aligned_a.append(a[i - 1])
            aligned_b.append('-')
            i -= 1
        elif direction == 3:
            aligned_a.append('-')
            aligned_b.append(b[j - 1])
            j -= 1
        else:
            break

    return AlignmentResult(
        ''.join(reversed(aligned_a)),
        ''.join(reversed(aligned_b)),
        best_score,
        mode,
        i,
        end_a,
        j,
        end_b,
    )

## 3. Readouts and Visualization

In [ ]:
def gap_runs(aligned):
    return [len(run) for run in re.findall(r'-+', aligned)]


def alignment_stats(result):
    a, b = result.aligned_a, result.aligned_b
    columns = len(a)
    matches = sum(x == y for x, y in zip(a, b) if x != '-' and y != '-')
    aligned_pairs = sum(x != '-' and y != '-' for x, y in zip(a, b))
    mismatches = sum(x != y and x != '-' and y != '-' for x, y in zip(a, b))
    gap_columns = sum(x == '-' or y == '-' for x, y in zip(a, b))
    gaps_a = gap_runs(a)
    gaps_b = gap_runs(b)
    all_gaps = gaps_a + gaps_b
    return {
        'score': result.score,
        'alignment_length': columns,
        'matches': matches,
        'mismatches': mismatches,
        'percent_identity_all_columns': 100 * matches / columns if columns else 0,
        'percent_identity_aligned_pairs': 100 * matches / aligned_pairs if aligned_pairs else 0,
        'gap_columns': gap_columns,
        'gap_runs': len(all_gaps),
        'gap_lengths': all_gaps,
        'longest_gap': max(all_gaps) if all_gaps else 0,
    }


def colored_alignment(result, width=80, max_columns=1200):
    a, b = result.aligned_a, result.aligned_b
    if len(a) > max_columns:
        a = a[:max_columns]
        b = b[:max_columns]
        note = f'<p><b>Showing first {max_columns} alignment columns.</b></p>'
    else:
        note = ''

    styles = {
        'match': 'background:#c7f2d0;color:#083b18;',
        'mismatch': 'background:#ffd6d6;color:#6f1111;',
        'gap': 'background:#d9e8ff;color:#123a6f;',
    }
    html = [note, '<pre style="font-family: ui-monospace, Consolas, monospace; line-height: 1.7; white-space: pre-wrap;">']
    for start in range(0, len(a), width):
        chunk_a = a[start:start + width]
        chunk_b = b[start:start + width]
        line_a = []
        line_mid = []
        line_b = []
        for x, y in zip(chunk_a, chunk_b):
            cls = 'gap' if x == '-' or y == '-' else ('match' if x == y else 'mismatch')
            line_a.append(f'<span style="{styles[cls]}">{x}</span>')
            line_mid.append('|' if x == y and x != '-' else ' ')
            line_b.append(f'<span style="{styles[cls]}">{y}</span>')
        html.append('A  ' + ''.join(line_a))
        html.append('   ' + ''.join(line_mid))
        html.append('B  ' + ''.join(line_b) + '\n')
    html.append('</pre>')
    display(HTML('\n'.join(html)))


def show_result(result, width=80):
    stats = alignment_stats(result)
    print(f"Mode: {result.mode}")
    print(f"Score: {stats['score']}")
    print(f"Alignment length: {stats['alignment_length']}")
    print(f"Percent identity, all columns: {stats['percent_identity_all_columns']:.2f}%")
    print(f"Percent identity, aligned pairs only: {stats['percent_identity_aligned_pairs']:.2f}%")
    print(f"Mismatches: {stats['mismatches']}")
    print(f"Gap columns: {stats['gap_columns']}")
    print(f"Gap runs: {stats['gap_runs']}; lengths: {stats['gap_lengths']}")
    print(f"Seq A coordinates: {result.start_a}..{result.end_a}")
    print(f"Seq B coordinates: {result.start_b}..{result.end_b}")
    colored_alignment(result, width=width)

## 4. Interactive Alignment App

Try changing the gap penalty from strongly negative to weakly negative. Watch whether one long gap turns into many smaller gaps, or whether unrelated ends get forced into a global alignment.

In [ ]:
PRESETS = {
    'Toy DNA: one deletion': {
        'type': 'DNA',
        'a': 'ACCGTATGCTTAGC',
        'b': 'ACGTTGCTAGC',
        'note': 'A short DNA example where a gap is expected.'
    },
    'Toy DNA: scattered-gap trap': {
        'type': 'DNA',
        'a': 'AAACCCGGGTTTAAACCC',
        'b': 'AAACCCGGGAAACCC',
        'note': 'Try gap = 0 to see why free gaps can make silly alignments.'
    },
    'Toy protein': {
        'type': 'Protein',
        'a': 'MEEPQSDPSVEPPLSQETFSDLWKLLPEN',
        'b': 'MEEPQSDLSIELPLSQETFSDLWKLLPEN',
        'note': 'A small protein example for Simple, BLOSUM62, or PAM250 scoring.'
    },
}

if widgets is not None:
    preset = widgets.Dropdown(options=list(PRESETS), value='Toy DNA: one deletion', description='Preset')
    seq_a_box = widgets.Textarea(description='Seq A', layout=widgets.Layout(width='95%', height='90px'))
    seq_b_box = widgets.Textarea(description='Seq B', layout=widgets.Layout(width='95%', height='90px'))
    note_html = widgets.HTML()
    mode_buttons = widgets.ToggleButtons(options=['global', 'local'], value='global', description='Mode')
    type_buttons = widgets.ToggleButtons(options=['DNA', 'Protein'], value='DNA', description='Type')
    matrix_dropdown = widgets.Dropdown(options=['Simple', 'BLOSUM62', 'PAM250'], value='Simple', description='Matrix')
    match_slider = widgets.IntSlider(value=2, min=-5, max=10, step=1, description='Match')
    mismatch_slider = widgets.IntSlider(value=-1, min=-10, max=5, step=1, description='Mismatch')
    gap_slider = widgets.IntSlider(value=-2, min=-15, max=0, step=1, description='Gap')
    width_slider = widgets.IntSlider(value=80, min=40, max=120, step=10, description='Width')
    run_button = widgets.Button(description='Run alignment', button_style='success', icon='check')
    output = widgets.Output()

    def load_preset(change=None):
        item = PRESETS[preset.value]
        seq_a_box.value = item['a']
        seq_b_box.value = item['b']
        type_buttons.value = item['type']
        note_html.value = f"<b>Preset note:</b> {item['note']}"

    def run_clicked(button=None):
        with output:
            output.clear_output()
            try:
                result = align(
                    seq_a_box.value,
                    seq_b_box.value,
                    mode=mode_buttons.value,
                    seq_type=type_buttons.value,
                    match=match_slider.value,
                    mismatch=mismatch_slider.value,
                    gap=gap_slider.value,
                    matrix_name=matrix_dropdown.value,
                )
                show_result(result, width=width_slider.value)
            except Exception as exc:
                display(HTML(f'<p style="color:#8a1f11;"><b>Could not run alignment:</b> {exc}</p>'))

    preset.observe(load_preset, names='value')
    run_button.on_click(run_clicked)
    load_preset()

    controls = widgets.VBox([
        widgets.HTML('<h3>Sequence Alignment App</h3><p>Pick a preset or paste your own sequences, choose scoring values, then click the button.</p>'),
        preset,
        note_html,
        seq_a_box,
        seq_b_box,
        widgets.HBox([mode_buttons, type_buttons, matrix_dropdown]),
        widgets.HBox([match_slider, mismatch_slider, gap_slider]),
        widgets.HBox([width_slider, run_button]),
        output,
    ])
    display(controls)
else:
    print('ipywidgets is not available, so here is one example alignment instead.')
    result = align(PRESETS['Toy DNA: one deletion']['a'], PRESETS['Toy DNA: one deletion']['b'], mode='global', match=2, mismatch=-1, gap=-2)
    show_result(result)

## 5. Good vs. Bad Parameter Story

Use this section to create screenshots for the write-up. The point is not that the good alignment always has the highest score, because the score changes when the scoring system changes. The point is that a biologically sensible scoring system should reward plausible evolutionary events.

In [ ]:
toy_a = 'AAACCCGGGTTTAAACCC'
toy_b = 'AAACCCGGGAAACCC'

print('Biologically sensible toy parameters: matches matter, gaps are costly but allowed')
good_toy = align(toy_a, toy_b, mode='global', match=2, mismatch=-2, gap=-3)
show_result(good_toy)

print('\nBiologically silly toy parameters: gaps are free, so the algorithm can scatter gaps too easily')
bad_toy = align(toy_a, toy_b, mode='global', match=2, mismatch=-2, gap=0)
show_result(bad_toy)

### My good-vs-bad interpretation

Replace this paragraph with your own observations. A good alignment should place homologous positions together, use gaps in a way that resembles real insertions/deletions, and avoid forcing unrelated sequence into the alignment just because the score system makes it cheap. A bad alignment can be technically optimal under its scoring rules but biologically silly if it explains the sequences using too many scattered gaps or aligns unrelated ends.

## 6. Fetch Real Sequences from NCBI

These helper functions fetch FASTA records from NCBI. If NCBI is unavailable during class, paste FASTA sequences into the app above instead.

In [ ]:
def fetch_fasta(accession, seq_start=None, seq_stop=None, database='nuccore'):
    base = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'
    params = {
        'db': database,
        'id': accession,
        'rettype': 'fasta',
        'retmode': 'text',
    }
    if seq_start is not None and seq_stop is not None:
        params['seq_start'] = str(seq_start)
        params['seq_stop'] = str(seq_stop)
    query = urllib.parse.urlencode(params)
    try:
        with urllib.request.urlopen(base + '?' + query, timeout=30) as response:
            text = response.read().decode('utf-8')
    except Exception as exc:
        raise RuntimeError(
            'Could not fetch from NCBI. Check your internet connection, then rerun this cell. '
            'If it still fails, paste FASTA sequences into the app manually.'
        ) from exc
    if not text.startswith('>'):
        raise RuntimeError(f'NCBI did not return a FASTA record for {accession}.')
    header = text.splitlines()[0]
    seq = ''.join(line.strip() for line in text.splitlines() if not line.startswith('>'))
    return header, seq.upper()


def print_fasta_info(name, header, seq):
    print(f'{name}: {len(seq)} characters')
    print(header)
    print(seq[:80] + ('...' if len(seq) > 80 else ''))

## 7. SARS-CoV vs. SARS-CoV-2 Spike: DNA and Protein

NCBI records used here:

- SARS-CoV Tor2 genome: `NC_004718.3`, spike coding region roughly `21492..25259`
- SARS-CoV-2 Wuhan-Hu-1 genome: `NC_045512.2`, spike coding region `21563..25384`
- SARS-CoV spike protein: `NP_828851.1`
- SARS-CoV-2 spike protein: `YP_009724390.1`

Run the DNA alignment first, then the protein alignment. DNA differences that disappear in the protein alignment are candidates for synonymous mutations.

In [ ]:
# DNA spike coding sequences
sars1_dna_header, sars1_spike_dna = fetch_fasta('NC_004718.3', 21492, 25259, database='nuccore')
sars2_dna_header, sars2_spike_dna = fetch_fasta('NC_045512.2', 21563, 25384, database='nuccore')

print_fasta_info('SARS-CoV spike DNA', sars1_dna_header, sars1_spike_dna)
print_fasta_info('SARS-CoV-2 spike DNA', sars2_dna_header, sars2_spike_dna)

In [ ]:
dna_spike_alignment = align(
    sars1_spike_dna,
    sars2_spike_dna,
    mode='global',
    seq_type='DNA',
    match=2,
    mismatch=-1,
    gap=-3,
)
show_result(dna_spike_alignment, width=100)

In [ ]:
# Protein spike sequences
sars1_prot_header, sars1_spike_protein = fetch_fasta('NP_828851.1', database='protein')
sars2_prot_header, sars2_spike_protein = fetch_fasta('YP_009724390.1', database='protein')

print_fasta_info('SARS-CoV spike protein', sars1_prot_header, sars1_spike_protein)
print_fasta_info('SARS-CoV-2 spike protein', sars2_prot_header, sars2_spike_protein)

In [ ]:
protein_spike_alignment = align(
    sars1_spike_protein,
    sars2_spike_protein,
    mode='global',
    seq_type='Protein',
    matrix_name='BLOSUM62',
    gap=-8,
)
show_result(protein_spike_alignment, width=100)

### DNA vs. protein interpretation

Write your answer here after running both alignments. Look for places where DNA bases differ but the protein alignment still has the same amino acid. Those are likely synonymous changes: the codon changed, but the encoded amino acid did not. Protein alignments focus attention on mutations that may change protein structure or function; DNA alignments also show silent changes and noncoding/codon-level details.

## 8. Optional: Translate DNA and Find Synonymous Differences

This simple translator helps connect DNA-level changes to amino-acid-level changes. It assumes the sequence starts in the correct reading frame.

In [ ]:
GENETIC_CODE = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','TCT':'S','TCC':'S','TCA':'S','TCG':'S','TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','TGT':'C','TGC':'C','TGA':'*','TGG':'W',
    'CTT':'L','CTC':'L','CTA':'L','CTG':'L','CCT':'P','CCC':'P','CCA':'P','CCG':'P','CAT':'H','CAC':'H','CAA':'Q','CAG':'Q','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','ACT':'T','ACC':'T','ACA':'T','ACG':'T','AAT':'N','AAC':'N','AAA':'K','AAG':'K','AGT':'S','AGC':'S','AGA':'R','AGG':'R',
    'GTT':'V','GTC':'V','GTA':'V','GTG':'V','GCT':'A','GCC':'A','GCA':'A','GCG':'A','GAT':'D','GAC':'D','GAA':'E','GAG':'E','GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}

def translate_dna(seq):
    seq = clean_sequence(seq).replace('U', 'T')
    aas = []
    for i in range(0, len(seq) - 2, 3):
        aas.append(GENETIC_CODE.get(seq[i:i+3], 'X'))
    return ''.join(aas)


def codon_comparison(seq_a, seq_b, max_rows=30):
    rows = []
    limit = min(len(seq_a), len(seq_b)) - 2
    for i in range(0, limit, 3):
        codon_a = seq_a[i:i+3]
        codon_b = seq_b[i:i+3]
        aa_a = GENETIC_CODE.get(codon_a, 'X')
        aa_b = GENETIC_CODE.get(codon_b, 'X')
        if codon_a != codon_b:
            rows.append((i // 3 + 1, codon_a, codon_b, aa_a, aa_b, aa_a == aa_b))
    print('codon\tSARS-CoV\tSARS-CoV-2\taa1\taa2\tsynonymous?')
    for row in rows[:max_rows]:
        print('\t'.join(map(str, row)))
    print(f'Showing {min(max_rows, len(rows))} of {len(rows)} differing codons.')


# After fetching the spike DNA sequences, run this to inspect codon-level changes.
# codon_comparison(sars1_spike_dna, sars2_spike_dna, max_rows=40)

## 9. Central Question: How Should We Quantify Alignment Quality?

My answer should include these ideas, rewritten in my own voice:

An alignment score is not an absolute measure of truth, because changing the match, mismatch, gap, or matrix values changes the score and sometimes changes the alignment. I would quantify alignment quality by combining the score with biological evidence: percent identity, conserved domains or motifs, whether gaps are rare and placed in plausible insertion/deletion regions, whether protein-changing mutations occur in meaningful functional regions, and whether the result agrees with known evolutionary relationships or a professional tool such as BLAST/EMBOSS. For coding DNA, I would also ask whether nucleotide changes are synonymous or nonsynonymous. A good alignment is the one that gives the most plausible evolutionary explanation, not just the largest number printed by one scoring system.

## 10. Stretch: Why Linear Gap Penalties Are Limited

This app uses a linear gap penalty: a gap of length 10 costs exactly ten times as much as a gap of length 1. Biology often does not behave that way. One insertion or deletion event can create a long gap, so a single long gap may be more plausible than many separate tiny gaps. A better model uses an affine gap penalty: one cost to open a gap and a smaller cost to extend it. That lets the aligner discourage many scattered gaps while still allowing one long realistic gap.

## Good parameters: sensible scoring




## Bad parameters: silly scoring




## Which alignment is right, and how would anyone know?

The right alignment is not automatically the one with the highest score, because the score is created by the scoring rules. If matches are rewarded, mismatches are penalized, and gaps have a realistic cost, the alignment is more likely to reflect real biological relationships. In the sensible-scoring run, gaps are expensive, so the algorithm avoids adding unnecessary insertions or deletions. In the silly-scoring run, the gap penalty is zero, so the algorithm can insert gaps for free and force more letters to line up. That may improve the score or percent identity, but it can make the alignment biologically less believable. A good alignment should line up homologous positions, keep conserved regions together, avoid scattered gaps, and explain differences as realistic mutations, insertions, or deletions. The score is useful, but the biological quality of the alignment is judged by percent identity, gap placement, mismatches, and whether the alignment makes sense for the sequences being compared.


## Further directions explored

Further directions explored: the tool was extended beyond toy DNA examples to include RNA and viral genome-style comparisons, including SARS-CoV, SARS-CoV-2, and a SARS-CoV-2 spike segment. These examples connect the alignment tool to real bioinformatics questions, such as tracking viral mutations and comparing related viruses.


# Submission Write-Up: Good vs. Bad Scoring

This section uses the sequence alignment website results to explain why scoring choices control the alignment.


## Good parameters: sensible scoring




## Bad parameters: silly scoring




## Which alignment is right, and how would anyone know?

The right alignment is not automatically the one with the highest score, because the score is created by the scoring rules. In the sensible-scoring run, matches are rewarded while mismatches and gaps both have meaningful penalties. This makes the alignment keep two differences as mismatches, which is biologically believable because those positions could represent mutations. In the silly-scoring run, the gap penalty is zero, so the algorithm can insert gaps for free and make the mismatches disappear. That alignment may look cleaner because it has fewer mismatches, but it is biologically worse because it explains simple substitutions as extra insertions and deletions. A good alignment should line up homologous positions, keep conserved regions together, avoid scattered gaps, and use gaps only when they make biological sense. The score helps choose an alignment, but percent identity, gap placement, mismatches, and biological plausibility decide whether the alignment is actually good.


## Further directions explored

Further directions explored: the tool was extended beyond basic DNA examples to include RNA and viral genome-style comparisons, including SARS-CoV, SARS-CoV-2, and a SARS-CoV-2 spike segment. These examples connect alignment scoring to real bioinformatics questions such as comparing related viruses, tracking mutations, and judging whether sequence differences are substitutions, insertions, or deletions.
